In [1]:
import os
import sounddevice as sd
from scipy.io.wavfile import write
import librosa
import torch
import numpy as np

In [2]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Tokenizer
from groq import Groq
from huggingface_hub import InferenceClient
from secret_api_keys import huggingface_api_key, groq_api_key
from langchain_core.prompts import PromptTemplate

c:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = huggingface_api_key

# 1. DeepSeek (HuggingFace)
deepseek_client = InferenceClient(
    model="deepseek-ai/DeepSeek-V3.2-Exp",
    token=huggingface_api_key
)

# 2. Meta Llama 3.1 (HuggingFace)
hf_llama_client = InferenceClient(
    model="meta-llama/Meta-Llama-3.1-8B-Instruct",
    token=huggingface_api_key
)

# 3. Groq Llama 3.1 8B Instant
groq_client = Groq(api_key=groq_api_key)


In [4]:
def generate_with(model_name, prompt):
    
    if model_name == "deepseek":
        resp = deepseek_client.chat_completion(
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.choices[0].message["content"]

    elif model_name == "hf_llama":
        resp = hf_llama_client.chat_completion(
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.choices[0].message["content"]

    elif model_name == "groq_llama":
        resp = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.choices[0].message.content

    else:
        raise ValueError("Unknown model")

In [5]:
prompt_template_for_title_suggestion = PromptTemplate(
    input_variables=["topic"],
    template="""
    I'm planning a blog post on the topic: {topic}.
    Suggest 10 catchy titles.
    Do NOT add explanations.
    """
)

In [6]:
tokenizer = Wav2Vec2Tokenizer.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'Wav2Vec2CTCTokenizer'. 
The class this function is called from is 'Wav2Vec2Tokenizer'.
c:\Python313\Lib\site-packages\transformers\models\wav2vec2\tokenization_wav2vec2.py:720: FutureWarning: The class `Wav2Vec2Tokenizer` is deprecated and will be removed in version 5 of Transformers. Please use `Wav2Vec2Processor` or `Wav2Vec2CTCTokenizer` instead.
  warnings.warn(
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
def record_audio(duration=5, filename="live.wav"):
    print("🎙️ Recording...")
    sr = 16000
    audio = sd.rec(int(duration * sr), samplerate=sr, channels=1, dtype='float32')
    sd.wait()
    write(filename, sr, audio)
    print("📁 Saved:", filename)

In [8]:
def speech_to_text(filename):
    print("🔍 Converting speech to text…")
    speech, sr = librosa.load(filename, sr=16000)

    input_values = tokenizer(speech, return_tensors="pt").input_values

    with torch.no_grad():
        logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)
    text = tokenizer.decode(predicted_ids[0])
    return text.lower()


In [9]:
record_audio(duration=5)

# 2️⃣ Convert Audio → Text
topic_text = speech_to_text("live.wav")
print("\n🎤 Recognized Topic:", topic_text)

# 3️⃣ Make Blog Title Prompt
prompt = prompt_template_for_title_suggestion.format(topic=topic_text)

# 4️⃣ Generate Titles via Groq (fastest)
print("\n⚡ Blog Titles:")
titles = generate_with("groq_llama", prompt)
print(titles)

# 5️⃣ Choose a title manually or auto
selected_title = input("\n✏️ Enter chosen title: ")

# 6️⃣ Generate Outline
outline_prompt = f"Create a detailed blog outline for: {selected_title}"
outline = generate_with("groq_llama", outline_prompt)

print("\n📝 Blog Outline:\n", outline)

# 7️⃣ Generate Full Blog
blog_prompt = f"Write a 1500-word blog titled '{selected_title}' using this outline:\n{outline}"
full_blog = generate_with("deepseek", blog_prompt)

print("\n📄 FULL BLOG:\n")
print(full_blog)

🎙️ Recording...
📁 Saved: live.wav
🔍 Converting speech to text…

🎤 Recognized Topic: ile my name is li

⚡ Blog Titles:
1. The Story Behind My Name: Li
2. I'm Li, Here's Everything You Need to Know
3. Li's Corner: A Journey Through Life
4. The Significance of My Name: Li
5. Hello, I'm Li: Get to Know Me
6. What's in a Name? My Story as Li
7. My Name's Li - A Blog About Life
8. I'm Li: The Person Behind the Name
9. Li's Life: A Personal Exploration
10. Meet Li: Discovering the Meaning of My Name

📝 Blog Outline:
 **Title: The Story Behind My Name: Li**

**I. Introduction**

- Brief overview of the significance of names in different cultures
- Explanation of why the author chose to explore the meaning behind their name
- Thesis statement: The name "Li" holds a rich history and cultural significance, reflecting the complexities and nuances of the Chinese and global identities.

**II. The Origins of Li**

- Background on the Chinese pronunciation system (Tone and characters)
- Explanation of

In [10]:
import sys
print("Notebook Python:", sys.executable)


Notebook Python: c:\Python313\python.exe
